# SE Queensland Regional Temperature Amplification

Computes the ratio of SE Queensland warming to global mean warming from the CMIP6 multi-model
ensemble. Used to scale entity global warming contributions (from FaIR) to the QLD region for
the 2022 floods liability calculation.

**Method**: stream CMIP6 `historical` surface air temperature (`tas`) from pangeo for 3 models.
Compute area-weighted annual means for (a) global and (b) SE Queensland. Fit linear trends
1901–2014. Amplification = QLD trend / global trend.

**SE QLD region**: lat −30° to −24°S, lon 150° to 154°E (Brisbane/Ipswich/Sunshine Coast catchment)  
**Baseline**: 1850–1900 (pre-industrial, same as notebook 02)  
**Trend period**: 1901–2014 (CMIP6 historical)

**ERA5 observed amplification** (wet-season Tmax vs FaIR GMST, α=0.289) is the PRIMARY input to
notebook 07, but is **not recomputed by any current notebook** — it is carried as the
`ERA5_observed` row in `qld_amplification_factor.csv` (1961–2020; trend 0.056°C/decade vs FaIR
GMST 0.195°C/decade). The save cell below **preserves** that row when rewriting the CSV so
reruns of this CMIP6-only notebook don't drop it. Reinstating a reproducible producer for it is
a tracked pipeline-hygiene follow-up (see `wiki/findings/2026-06-17-lei-dropna-fix.md`).

> **Status note (2026-06-13 methodology revision).** This notebook's α_QLD is an **annual-mean
> `tas`** ratio across 2 CMIP6 models; the p05/p95 are interpolation between two points, not a
> sampling range. It is a *sensitivity* input only. The final liability uses each entity's
> **global** warming share directly — the `warming_qld_*` columns written here are **diagnostic
> only**. α_QLD enters the Probability Ratio as the multiplicative shift coefficient
> β = ln(1+CC_rate)·α in the GEV shift-fit (notebook 07): the ERA5 α=0.289 is the conservative
> primary, this CMIP6 α=0.882 is a sensitivity.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import intake
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROC = Path('../../data/processed')
FIGS = Path('../../outputs/figures')

# SE Queensland bounding box (Brisbane/Ipswich/Sunshine Coast catchment)
LAT_S, LAT_N = -30, -24
LON_W, LON_E = 150, 154

# Baseline and trend periods — same as notebook 02
BASELINE = slice('1850', '1900')
TREND    = slice('1901', '2014')

# Same models as notebook 02 — Australian models preferred for regional skill
MODELS = ['ACCESS-CM2', 'ACCESS-ESM1-5', 'MPI-ESM1-2-HR']

## 1. Helper functions (identical to notebook 02)

In [ ]:
def area_weighted_mean(ds, var='tas', lat_s=None, lat_n=None, lon_w=None, lon_e=None):
    """Compute cosine-latitude weighted spatial mean, optionally over a bounding box."""
    da = ds[var]
    extra_dims = [d for d in da.dims if d not in ('time', 'lat', 'lon')]
    if extra_dims:
        da = da.isel({d: 0 for d in extra_dims})
    if lon_w is not None:
        if da.lon.values.min() < 0:
            da = da.assign_coords(lon=(da.lon % 360)).sortby('lon')
        lon_w_norm = lon_w % 360
        lon_e_norm = lon_e % 360
        da = da.sel(lat=slice(lat_s, lat_n), lon=slice(lon_w_norm, lon_e_norm))
    elif lat_s is not None:
        da = da.sel(lat=slice(lat_s, lat_n))
    weights = np.cos(np.deg2rad(da.lat)).broadcast_like(da)
    return da.weighted(weights).mean(dim=['lat', 'lon']).squeeze()


def annual_anomaly(ts, baseline=BASELINE):
    """Resample monthly to annual, compute anomaly relative to baseline."""
    annual = ts.resample(time='YE').mean()
    base_mean = annual.sel(time=baseline).mean('time')
    return annual - base_mean


def linear_trend(ts, period=TREND):
    """Return OLS trend (°C/year) over the specified period as a Python float."""
    sub = ts.sel(time=period).dropna('time')
    vals = np.asarray(sub.values).flatten()
    years = sub.time.dt.year.values.astype(float)
    slope, _, _, _, _ = stats.linregress(years, vals)
    return float(slope)

## 2. Load CMIP6 catalog and fetch model data

In [ ]:
CATALOG_LOCAL = Path('../../data/processed/pangeo-cmip6.json').resolve()

print(f'Loading catalog from local cache: {CATALOG_LOCAL}')
col = intake.open_esm_datastore(str(CATALOG_LOCAL))

cat = col.search(
    variable_id='tas',
    experiment_id='historical',
    table_id='Amon',
    source_id=MODELS,
    member_id='r1i1p1f1',
)
print(f'Found {len(cat.df)} entries:')
print(cat.df[['source_id', 'member_id', 'grid_label', 'zstore']].to_string(index=False))

In [ ]:
results = []

for _, row in cat.df.iterrows():
    model = row['source_id']
    zstore = row['zstore']
    print(f'Processing {model}...', end=' ', flush=True)
    try:
        ds = xr.open_zarr(zstore, consolidated=True)

        ts_global = area_weighted_mean(ds)
        anom_global = annual_anomaly(ts_global)
        trend_global = linear_trend(anom_global)

        ts_qld = area_weighted_mean(ds, lat_s=LAT_S, lat_n=LAT_N, lon_w=LON_W, lon_e=LON_E)
        anom_qld = annual_anomaly(ts_qld)
        trend_qld = linear_trend(anom_qld)

        amplification = float(trend_qld / trend_global)

        results.append({
            'model': model,
            'trend_global_degC_per_yr': trend_global,
            'trend_qld_degC_per_yr': trend_qld,
            'amplification': amplification,
            'anom_global': anom_global,
            'anom_qld': anom_qld,
        })
        print(f'amplification = {amplification:.3f}')
    except Exception as e:
        print(f'FAILED: {e}')

print(f'\nSuccessfully processed {len(results)}/{len(cat.df)} models')

## 3. Ensemble amplification factor

In [ ]:
amps = pd.DataFrame([{
    'model': r['model'],
    'trend_global': r['trend_global_degC_per_yr'] * 100,   # °C/century
    'trend_qld':    r['trend_qld_degC_per_yr'] * 100,
    'amplification': r['amplification'],
} for r in results])

amp_p50 = float(np.median(amps['amplification']))
amp_p05 = float(np.percentile(amps['amplification'], 5))
amp_p95 = float(np.percentile(amps['amplification'], 95))

# Compare to SE AU from notebook 02
AU_AMP_CMIP6 = 0.935

print('SE Queensland amplification factor (CMIP6 historical annual tas, 1901–2014):')
print(f'  Median (p50): {amp_p50:.3f}')
print(f'  5th–95th:     [{amp_p05:.3f}, {amp_p95:.3f}]')
print(f'  SE AU (notebook 02): {AU_AMP_CMIP6:.3f}')
print()
print('Per-model breakdown:')
print(amps[['model', 'trend_global', 'trend_qld', 'amplification']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for r in results:
    anom_qld = r['anom_qld'].sel(time=TREND)
    years = anom_qld.time.dt.year.values
    ax.plot(years, anom_qld.values, alpha=0.6, linewidth=1, label=r['model'])
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_title('SE QLD temperature anomaly — CMIP6 historical', fontsize=11)
ax.set_xlabel('Year')
ax.set_ylabel('°C anomaly (vs 1850–1900)')
ax.legend(fontsize=8)

ax2 = axes[1]
colors = ['#FF5722' if 'ACCESS' in m else '#2196F3' for m in amps['model']]
ax2.barh(amps['model'], amps['amplification'], color=colors, alpha=0.8)
ax2.axvline(amp_p50, color='k', linewidth=1.5, linestyle='--',
            label=f'QLD median = {amp_p50:.2f}')
ax2.axvline(AU_AMP_CMIP6, color='grey', linewidth=1.2, linestyle=':',
            label=f'SE AU = {AU_AMP_CMIP6:.2f}')
ax2.axvspan(amp_p05, amp_p95, alpha=0.1, color='k',
            label=f'5–95th [{amp_p05:.2f}, {amp_p95:.2f}]')
ax2.axvline(1.0, color='lightgrey', linewidth=0.8, linestyle=':')
ax2.set_xlabel('Amplification factor (QLD / global annual tas trend)')
ax2.set_title('SE QLD warming amplification by model', fontsize=11)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGS / 'qld_amplification.png', bbox_inches='tight')
plt.show()

## 4. Apply to entity warming contributions

Add QLD-specific warming columns to `entity_warming_contribution.parquet`. The amplification
factor scales the *magnitude* of warming attributable to each entity in the QLD context.
Entity proportional shares within Carbon Majors are unchanged.

In [ ]:
ew = pd.read_parquet(PROC / 'entity_warming_contribution.parquet')

ew['warming_qld_p50_degC'] = ew['warming_p50_degC'] * amp_p50
ew['warming_qld_p05_degC'] = ew['warming_p05_degC'] * amp_p05
ew['warming_qld_p95_degC'] = ew['warming_p95_degC'] * amp_p95

print(f'QLD amplification applied: {amp_p50:.3f} [p05={amp_p05:.3f}, p95={amp_p95:.3f}]')
print(f'SE AU amplification (nb02): {AU_AMP_CMIP6:.3f}')
print()
print('Top 10 entities — SE QLD attributed warming (°C, p50):')
top10 = ew.nlargest(10, 'warming_qld_p50_degC')[[
    'parent_entity', 'parent_type', 'warming_p50_degC', 'warming_qld_p50_degC'
]]
top10['warming_p50_mdegC']     = top10['warming_p50_degC'] * 1000
top10['warming_qld_p50_mdegC'] = top10['warming_qld_p50_degC'] * 1000
print(top10[['parent_entity', 'warming_p50_mdegC', 'warming_qld_p50_mdegC']].to_string(index=False))

## 5. Save outputs

In [ ]:
ew.to_parquet(PROC / 'entity_warming_contribution.parquet', index=False)

amps_out = amps[['model', 'trend_global', 'trend_qld', 'amplification']].copy()

# Preserve externally-produced rows already in the CSV — notably the ERA5_observed wet-season
# Tmax amplification (α=0.289), which is the PRIMARY input to notebook 07 and is NOT recomputed
# in this CMIP6 notebook. Without this guard, rerunning nb06 silently drops that row and breaks
# notebook 07 (KeyError on qld_af.loc['ERA5_observed']). See wiki/findings/2026-06-17-lei-dropna-fix.md.
csv_path = PROC / 'qld_amplification_factor.csv'
if csv_path.exists():
    prior = pd.read_csv(csv_path)
    preserved = prior[~prior['model'].isin(amps_out['model'])]
    if len(preserved):
        print(f'Preserving {len(preserved)} externally-produced row(s): {list(preserved["model"])}')
        amps_out = pd.concat([amps_out, preserved], ignore_index=True)

amps_out.to_csv(csv_path, index=False)

print('Saved:')
print(f'  entity_warming_contribution.parquet — added warming_qld_p05/p50/p95_degC columns')
print(f'  qld_amplification_factor.csv        — per-model amplification breakdown')
print()
print('Contents of qld_amplification_factor.csv:')
print(amps_out.to_string(index=False))
print()
print(f'Figures saved to: {FIGS / "qld_amplification.png"}')

## Key findings

| Source | α_QLD | Notes |
|--------|-------|-------|
| ACCESS-CM2 (CMIP6 historical) | 0.364 | Low — QLD warms much less than global in this model |
| ACCESS-ESM1-5 (CMIP6 historical) | 1.401 | High — QLD warms faster than global |
| **CMIP6 ensemble median** | **0.882** | p05=0.416, p95=1.349 (only 2 models; MPI-ESM1-2-HR unavailable on pangeo) |
| **ERA5 observed (wet-season Tmax)** | **0.289** | 1961–2020; used as primary in notebook 07 — conservative lower bound |

Compare SE Australia (notebooks 02 + 05): CMIP6=0.935, ERA5=0.726. QLD wet-season Tmax shows a much weaker trend (0.056°C/decade) than the global FaIR trend (0.195°C/decade), likely due to ENSO variability and cloud cover feedback.

- ERA5 α_QLD = 0.289 is conservative for precipitation attribution — land Tmax underestimates the SST-driven moisture forcing relevant to QLD flood extremes. True CC scaling factor likely higher (SST amplification ~1.0).
- `warming_qld_p05/p50/p95_degC` columns appended to `entity_warming_contribution.parquet` using CMIP6 median α=0.882.
- Carbon Majors total QLD-attributed warming (p50): 464.3 m°C [p05=161.3, p95=944.6].

→ See `wiki/findings/2026-05-26-qld-floods-regional-amplification.md` for the full write-up.